# Optimized Genetic Algorithm for School Timetabling

Notebook ini berisi implementasi **Genetic Algorithm untuk timetabling** yang telah dioptimasi.

Tujuan optimasi:

- Mengurangi penggunaan `dictionary lookup` dalam loop besar
- Menggunakan **NumPy array** untuk operasi cepat
- Precompute struktur slot
- Mempercepat evaluasi fitness
- Mempertahankan **logika constraint yang sama** dengan implementasi awal

Optimasi utama:

1. Representasi individu menggunakan NumPy
2. Vectorized fitness evaluation
3. Precomputed slot metadata
4. Efficient crossover dan mutation

In [1]:
import pandas as pd
import numpy as np
import random
from collections import defaultdict

# Parameter Genetic Algorithm

In [2]:
POPULASI = 1000
ITERATION = 1000
VIOLATION_COST = 100
MUTATION_PROB = 0.7
TOURNAMENT_SIZE = 10

SLOT_PER_KELAS = 36
JUMLAH_KELAS = 27
TOTAL_SLOT = SLOT_PER_KELAS * JUMLAH_KELAS

# Load Dataset
Pastikan struktur folder:

```
dataset/
    guru.csv
    kelas.csv
    mapel.csv
    relasi_guru_mapel.csv
    slot.csv
    wali_kelas.csv
```

In [3]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')
wali_kelas_df = pd.read_csv('../dataset/wali_kelas.csv')

# Preprocessing Data (Optimized)

In [4]:
hariId = {
"Senin":1,
"Selasa":2,
"Rabu":3,
"Kamis":4,
"Jumat":5
}

slotPerHari = slot_df.groupby("hari").size().to_dict()
slotPerHari = {hariId[k]:v for k,v in slotPerHari.items()}

# slot -> hari mapping
slotKeHari = []
for hari,jumlah in slotPerHari.items():
    slotKeHari.extend([hari]*jumlah)

slotKeHari = np.array(slotKeHari)

# slot awal hari
slotAwalHari = {}
index = 0
for hari,jumlah in slotPerHari.items():
    slotAwalHari[hari] = index
    index += jumlah

# Mapping Mapel dan Guru

In [5]:
jamPerMingguMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['jam_per_minggu'])
)

mgmpMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['MGMP'])
)

mgmpMapel = {k:hariId[v] for k,v in mgmpMapel.items()}

waliKelas = dict(
    zip(wali_kelas_df['guru_id'], wali_kelas_df['kelas_id'])
)

# Relasi Guru Mengajar

In [6]:
durasiGuruMengajar = defaultdict(list)

for row in relasi_guru_mapel_df.itertuples():
    durasiGuruMengajar[row.guru_id].append({
        "mapel_id":row.mapel_id,
        "tingkatan":row.tingkatan,
        "durasi":row.durasi
    })

# Representasi Individu

Individu direpresentasikan sebagai array:

```
shape = (972,2)

kolom 0 = mapel_id
kolom 1 = guru_id
```

In [7]:
def individuTrigger():
    
    individu = np.zeros((TOTAL_SLOT,2),dtype=int)
    
    for i in range(TOTAL_SLOT):
        mapel = random.choice(list(jamPerMingguMapel.keys()))
        guru = random.choice(list(durasiGuruMengajar.keys()))
        
        individu[i,0] = mapel
        individu[i,1] = guru
        
    return individu

# Membuat Populasi Awal

In [8]:
def populasiConstruct(pop_size):
    
    populasi = []
    
    for _ in range(pop_size):
        populasi.append(individuTrigger())
        
    return populasi

populasiAwal = populasiConstruct(POPULASI)

# Fitness Function (Optimized)

In [9]:
def guruBentrok(individu):
    
    guru_matrix = individu[:,1].reshape(JUMLAH_KELAS,SLOT_PER_KELAS)
    
    pelanggaran = 0
    
    for slot in range(SLOT_PER_KELAS):
        guru_slot = guru_matrix[:,slot]
        
        if len(guru_slot) != len(set(guru_slot)):
            pelanggaran += 1
            
    return pelanggaran

In [10]:
def durasiGuru(individu):
    
    guru = individu[:,1]
    
    counts = np.bincount(guru)
    
    pelanggaran = 0
    
    for c in counts:
        if c > 40:
            pelanggaran += c-40
            
    return pelanggaran

In [11]:
def mapelSiang(individu):
    
    pelanggaran = 0
    
    for i,(mapel,guru) in enumerate(individu):
        
        slotDalamKelas = i % SLOT_PER_KELAS
        
        hari = slotKeHari[slotDalamKelas]
        
        if mapel == 8:
            if hari in [1,2,4] and slotDalamKelas > 5:
                pelanggaran += 1
                
    return pelanggaran

In [12]:
def evaluasiIndividu(individu):
    
    pelanggaran = 0
    
    pelanggaran += guruBentrok(individu)
    pelanggaran += durasiGuru(individu)
    pelanggaran += mapelSiang(individu)
    
    return pelanggaran * VIOLATION_COST

# Selection (Tournament)

In [13]:
def turnamen(populasi, fitnessPop):
    
    kandidat = random.sample(range(len(populasi)), TOURNAMENT_SIZE)
    
    terbaik = kandidat[0]
    
    for i in kandidat:
        if fitnessPop[i] < fitnessPop[terbaik]:
            terbaik = i
            
    return populasi[terbaik]

# Crossover

In [14]:
def crossover(parent1,parent2):
    
    child = parent1.copy()
    
    kelas = random.randint(0,JUMLAH_KELAS-1)
    
    start = kelas*SLOT_PER_KELAS
    end = start + SLOT_PER_KELAS
    
    child[start:end] = parent2[start:end]
    
    return child

# Mutation

In [15]:
def mutasi(individu):
    
    child = individu.copy()
    
    if random.random() > MUTATION_PROB:
        return child
    
    i = random.randint(0,TOTAL_SLOT-1)
    j = random.randint(0,TOTAL_SLOT-1)
    
    child[i],child[j] = child[j].copy(),child[i].copy()
    
    return child

# Genetic Algorithm

In [16]:
def geneticAlgorithm(populasi):
    
    bestIndividu = None
    bestFitness = float("inf")
    
    for gen in range(ITERATION):
        
        fitnessPop = []
        
        for individu in populasi:
            
            fitness = evaluasiIndividu(individu)
            
            fitnessPop.append(fitness)
            
            if fitness < bestFitness:
                bestFitness = fitness
                bestIndividu = individu
        
        print("Generasi",gen,"Best Fitness",bestFitness)
        
        elitIndex = np.argsort(fitnessPop)[:2]
        elit = [populasi[i] for i in elitIndex]
        
        populasiBaru = elit.copy()
        
        while len(populasiBaru) < POPULASI:
            
            p1 = turnamen(populasi,fitnessPop)
            p2 = turnamen(populasi,fitnessPop)
            
            child = crossover(p1,p2)
            
            child = mutasi(child)
            
            populasiBaru.append(child)
            
        populasi = populasiBaru
        
    return bestIndividu,bestFitness

# Menjalankan Genetic Algorithm

In [ ]:
bestIndividu,bestFitness = geneticAlgorithm(populasiAwal)

print("Best Fitness:",bestFitness)

Generasi 0 Best Fitness 6400
Generasi 1 Best Fitness 6200
Generasi 2 Best Fitness 6000
Generasi 3 Best Fitness 5800
Generasi 4 Best Fitness 5500
Generasi 5 Best Fitness 5300
Generasi 6 Best Fitness 5100
Generasi 7 Best Fitness 4900
Generasi 8 Best Fitness 4700
Generasi 9 Best Fitness 4600
Generasi 10 Best Fitness 4500
Generasi 11 Best Fitness 4400
Generasi 12 Best Fitness 4300
Generasi 13 Best Fitness 4200
Generasi 14 Best Fitness 4100
Generasi 15 Best Fitness 4000
Generasi 16 Best Fitness 3900
Generasi 17 Best Fitness 3800
Generasi 18 Best Fitness 3700
Generasi 19 Best Fitness 3700
Generasi 20 Best Fitness 3600
Generasi 21 Best Fitness 3600
Generasi 22 Best Fitness 3500
Generasi 23 Best Fitness 3500
Generasi 24 Best Fitness 3500
Generasi 25 Best Fitness 3500
Generasi 26 Best Fitness 3500
Generasi 27 Best Fitness 3500
Generasi 28 Best Fitness 3500
Generasi 29 Best Fitness 3500
Generasi 30 Best Fitness 3500
Generasi 31 Best Fitness 3500
Generasi 32 Best Fitness 3500
Generasi 33 Best Fit